# Homework 1 Full Solutions
**Notebook:** Detailed solutions to all HW1 problems with circuit implementations and measurements

## Structure

| Section | Content |
|---------|---------|
| **HW1 Problem 1** | Phase extraction, prepare $\psi $ with $U$ gate, measure in $Z$ and $X$ basises |
| **HW1 Problem 2** | Alternative preparation using $H$, $R_z$ decomposition; real-device comparison |
| **HW1 Problem 3** | Gate identity proofs ($HXH=Z$, CZ symmetry, CNOT conjugation) |
| **HW1 Problem 4** | Sequential measurement probability equivalence |

### Homework 1

 <b>Problem 1: Extracting Qubit phase</b>
 
 
 Suppose that we have the Qubit in the state 
  \begin{align} \label{Eq:1} \tag{1} 
  |\psi\rangle = \sin\theta|0\rangle + e^{i\varphi}\cos\theta|1\rangle
  \end{align}
  Show how using the measurement in computational basis 
 to measure the angle $\varphi$ provided $\theta$ is known.
    Write the code using Qiskit which prepares this state 
  and performs this measurement. Visualise the circuit. 
  Give link to jupyter notebook. 
  
  <b> Solution </b>
  Let's apply Hadamard gate to the state (1)
 
 $ H|\psi\rangle = \frac{(\sin\theta + e^{i\varphi} \cos\theta )}{\sqrt{2}}|0\rangle + 
  \frac{(\sin\theta - e^{i\varphi} \cos\theta )}{\sqrt{2}} \cos\theta|1\rangle $
  
  Measuerement after applying the Hadamard gate is called measurement in the superposition basis. 
  Then, measurement of the state $\psi$ in superposition basis will give the probabilities $(1\pm \cos\varphi\sin(2\theta))/2$ which allows determining both angles $\theta$ and $\varphi$. 


<b> Now let's implement that using qiskit. </b> 

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

import numpy as np
import qiskit as qk
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_bloch_multivector, plot_histogram
from math import pi

# --- IBM Quantum hardware setup (optional) ---
#from qiskit_ibm_runtime import QiskitRuntimeService
#from qiskit_ibm_runtime import SamplerV2 as Sampler
#from collections import Counter

#service = QiskitRuntimeService(channel='ibm_cloud', token=os.environ['IBM_QUANTUM_TOKEN'])'
#backend_real = service.least_busy(operational=True, simulator=False, min_num_qubits=5)
#sampler = Sampler(backend_real)

Use the custom $U_3$ gate to prepare the required state with $\theta=\varphi=\pi/3$

In [ ]:
circuit = QuantumCircuit(1, 1)
circuit.u(-pi / 3, pi, 0, 0)   # u3 renamed to u in Qiskit 1.x (same parameters)
circuit.draw(output='mpl')

In [ ]:
circuit.measure(range(1), range(1))
backend = AerSimulator()
resultZSim = backend.run(transpile(circuit, backend), shots=1024).result()
plot_histogram([resultZSim.get_counts(circuit)])

Measuring in the superposition basis gives the coeffitients which ideally should be  $(1 + \cos\varphi\sin(2\theta))/2=0.716$ and $(1 - \cos\varphi\sin(2\theta))/2=0.284$

In [ ]:
circuit1 = QuantumCircuit(1, 1)
# Prepare same state |psi> = sin(theta)|0> + exp(i*phi)*cos(theta)|1> but measured in X-basis:
# Apply H after state prep to rotate the X-basis onto the Z-basis before measuring.
# X-basis measurement: gives P(|0>) = (1 + cos(phi)*sin(2*theta))/2 ≈ 0.716
#                                     P(|1>) = (1 - cos(phi)*sin(2*theta))/2 ≈ 0.284
circuit1.u(-pi / 3, pi + pi / 3, 0, 0)   # u3 â†’ u in Qiskit 1.x
circuit1.h(range(1))                       # H rotates X-basis to Z-basis: reveals relative phase phi
circuit1.measure(range(1), range(1))
backend = AerSimulator()
resultZSim = backend.run(transpile(circuit1, backend), shots=1024).result()
plot_histogram([resultZSim.get_counts(circuit1)])

In [ ]:
circuit1 = QuantumCircuit(1, 1)
circuit1.u(-pi / 3, pi + pi / 3, 0, 0)
circuit1.h(range(1))
circuit1.measure(range(1), range(1))
backend = AerSimulator()
resultZSim = backend.run(transpile(circuit1, backend), shots=1024).result()
plot_histogram([resultZSim.get_counts(circuit1)])

<b> Problem 2: Single Qubit state preparation </b>
    
  Suggest a circuit which uses fixed number of standard gates: $X$, $H$, $S$ and $R_z(\gamma)$
  where $R_z(\gamma)$ is rotation around $z$ axis by an arbitrary angle $\gamma$ 
  to prepare the state $|\psi\rangle = \sin\theta|0\rangle + e^{i\varphi}\cos\theta|1\rangle$ modulo the total phase of the wave function. 
  Compare the accuracy of this procedure  with custom $U_3$ gate at IMB Quantum
  for the angles $\varphi=\theta = \pi/3$. 
    Write the code using Qiskit, give link to jupyter notebook. 
  
<b> Solution </b> 
  We can search for the  transformation using the following chain of transforms 
  
  $H |0\rangle = (|0\rangle + |1\rangle )/\sqrt{2}$, 
  
  $R_z(2 \theta  + \pi) H |0\rangle = (|0\rangle - e^ {2i \theta } |1\rangle )/\sqrt{2} $,
  
  $H R_z(2 \theta+ \pi) H |0\rangle = -i e^ {i \theta } (\sin\theta|0\rangle + i  \cos \theta|1\rangle )$,
  
  $R_z(\varphi-\pi/2) H R_z(2 \theta+ \pi) H |0\rangle =  -i e^ {i \theta } (\sin \theta|0\rangle + e^{i\varphi} \cos \theta |1\rangle )$

  Below we check this state by comparing with the actions if custom $U_3$ operator discussed above.
  
  <br />
<br />
  


**Problem 2: H–Rz–H decomposition measured in Z-basis**

Circuit implements: $R_z(\phi - \pi/2)\, H\, R_z(2\theta + \pi)\, H\, |0\rangle$

which produces $-i\,e^{i\theta}\bigl(\sin\theta\,|0\rangle + e^{i\phi}\cos\theta\,|1\rangle\bigr)$ up to global phase.

Z-basis: should give $P(|0\rangle) \approx 0.75\ (= \sin^2(\pi/3))$, $P(|1\rangle) \approx 0.25\ (= \cos^2(\pi/3))$

In [ ]:
circuit1 = QuantumCircuit(1, 1)
circuit1.h(range(1))
circuit1.rz(pi * 2 / 3 + pi, range(1))
circuit1.h(range(1))
circuit1.rz(pi / 3 - pi / 2, range(1))
circuit1.measure(range(1), range(1))
backend = AerSimulator()
resultZSim = backend.run(transpile(circuit1, backend), shots=1024).result()
plot_histogram([resultZSim.get_counts(circuit1)])

<b> Problem 2: H-Rz-H decomposition measured in X-basis (add H before measure) </b>

# Problem 2: H-Rz-H decomposition measured in X-basis (add H before measure)
# X-basis: should give P(|0>) ≈ 0.716, P(|1>) ≈ 0.284 â€” same as U gate result

In [ ]:
circuit1 = QuantumCircuit(1, 1)
circuit1.h(range(1))
circuit1.rz(pi * 2 / 3 + pi, range(1))
circuit1.h(range(1))
circuit1.rz(pi / 3 - pi / 2, range(1))
circuit1.h(range(1))
circuit1.measure(range(1), range(1))
backend = AerSimulator()
resultZSim = backend.run(transpile(circuit1, backend), shots=1024).result()
plot_histogram([resultZSim.get_counts(circuit1)])

Let us compare how well the operators  $U_3(2\pi/3, \pi/3, -\pi/3)$ and $R_z(\varphi-\pi/2) H R_z(2 \theta+ \pi) H $ prepare the state on the real quantum computer. 

In [ ]:
backend = AerSimulator()

circuitU3 = QuantumCircuit(1, 1)
circuitU3.u(-pi / 3, pi + pi / 3, 0, 0)   # u3 â†’ u in Qiskit 1.x
circuitU3.measure(range(1), range(1))

circuitRHRH = QuantumCircuit(1, 1)
circuitRHRH.h(range(1))
circuitRHRH.rz(pi * 2 / 3 + pi, range(1))
circuitRHRH.h(range(1))
circuitRHRH.rz(pi / 3 - pi / 2, range(1))
circuitRHRH.measure(range(1), range(1))

# Simulator comparison
job = backend.run(transpile([circuitU3, circuitRHRH], backend), shots=1024)
result = job.result()
resultSimU3   = result.get_counts(0)
resultSimRHRH = result.get_counts(1)



# Real device (uncomment when IBM Quantum account is available):

#from qiskit_ibm_runtime import QiskitRuntimeService
#from qiskit_ibm_runtime import SamplerV2 as Sampler
#from collections import Counter

#service = QiskitRuntimeService(channel='ibm_cloud', token=os.environ['IBM_QUANTUM_TOKEN'])
#backend_real = service.least_busy(operational=True, simulator=False, min_num_qubits=5)
#sampler = Sampler(backend_real)

#job_real = sampler.run([transpile(circuitU3, backend_real),
#                         transpile(circuitRHRH, backend_real)], shots=1024)
#resultSimU3   = job_real.result()[0].data.c.get_counts()
#resultSimRHRH = job_real.result()[1].data.c.get_counts()

#plot_histogram([resultSimU3, resultSimRHRH],
#               legend=['u gate', 'H-Rz-H decomposition'])

<b> Problem 3: Useful identities for 1-Qubit and 2-Qubit gates </b>
  
  Prove the following identities 1) $HXH=Z$, 2) $cZ_{12}=cZ_{21}$, 
  3) $H_1 H_2 cX_{12} H_1 H_2 = cX_{21}$, 4) $ce^{i\alpha}_{12} =  U_{1}(\alpha ) $. 
  
  <b> Solution </b> 
   1) $HXH = (\sigma_x + \sigma_z) \sigma_x (\sigma_x + \sigma_z)/2 =
  \sigma_x ( \sigma_x - \sigma_z) (\sigma_x + \sigma_z)/2 = \sigma_z  $- 
  
   2) We can use that $\sigma_z |0\rangle = |0\rangle$ so that $cZ_{12} 
    |0\rangle_1|0\rangle_2  = cZ_{21}  |0\rangle_1|0\rangle_2  $, 
   $cZ_{12}  |1\rangle_1|0\rangle_2  = cZ_{21}  |1\rangle_1|0\rangle_2  $ and so on for all other basis elements in 2-Qubit space. 
   
   3 ) This follows from 1) and 2)
   
   4) $c e^{i\alpha}_{12}  |1\rangle_1|0\rangle_2 = e^{i\alpha}|1\rangle_1|0\rangle_2 $ 
   and
 $c e^{i\alpha}_{12}  |0\rangle_1|0\rangle_2 =|0\rangle_1|0\rangle_2 $ .
 This coincides with the action of the operator $ U_1(\alpha) =\left( \begin{smallmatrix}
   1 & 0 \\ 0 & e^{i\alpha} \end{smallmatrix} \right ) $

  
